#  V tomto modulu upravíme si data, než budeme z nich  zjištovat opravdu něco užitečného

In [1]:
import pandas as pd

# Načteme data, přejmenujeme sloupce

In [2]:
original_columns_names = [
    'url','name','price','brand', 'colors','motor power','battery power capacity','range','speed'
]

better_columns_names = [
    'url','name','price','brand', 'colors', 'motor_power_w','battery_capacity_wh','range_km','speed_kmh'
]

parsed_data = pd.read_csv(
    './data.tsv', 
    sep='\t', 
    encoding='utf-16',
    index_col=False,
    header=0,
    names=better_columns_names 
)
parsed_data = parsed_data.dropna(how='all')
parsed_data.tail(3)

,url,name,price,brand,colors,motor_power_w,battery_capacity_wh,range_km,speed_kmh
322,https://www.geekbuying.com/item/TWOFISH-TW4-PR...,"TWOFISH TW4 PRO Electric Scooter, 2*1000W Moto...",19955.20,NaN,NaN,0,0Wh,40.0,72.0
323,https://www.geekbuying.com/item/Magicycle-Ocel...,"Magicycle Ocelot Pro Electric Bike, 750W Motor...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
324,"Disc Brakes, Hydraulic Lockout Suspension Fork...",41720.78,Magicycle,White,1500,",",100,28.0,NaN


# 0. URL

- **Problém**: chceme dodatečně zkontrolovat formaty `url` - nezajímají nás totíž data nevalidní řádky
- **Řešení**: pomocí regulárních výrazů zbavíme se špatných řádků.


In [3]:
print(f"Number of rows before filtering: {len(parsed_data)}", end='\n\n')

url_pattern = r"^https?://"
valid_urls = parsed_data["url"].astype("string").str.match(url_pattern)
parsed_data = parsed_data[valid_urls]

print(f"Number of rows after filtering: {len(parsed_data)}", end='\n\n')

Number of rows before filtering: 323

Number of rows after filtering: 320



# 1. Kvalitativní data (color)
- **Problém**: `colors` může v sobě mít několik barev rozdělených nekonzistentně (LLM vyplnění se trochu liší)
- **Řešení**: zjistíme všechny barvy, zduplikujeme data - stejný řádek pro každou další barvu daného zboží. 

**Proč**: 

Následující úlohy `(klasterizace` a `predikce`) i když známe, a mohli bychom s daty teď nějak manipulovat - bude to ale spíš otázka dalších zadání (`task_2_1 / task_2_2`). 

V tomto zadání zůstaňme co nejvíc objektivní a přímočařejší

alternativa např. `one hot encoding` -> `míň duplikací` - mohlo by to ale negativně hrát během cenové predikce. 

### Provedeme analýzu sloupce `colors`

- vidíme i hodně jednotlivých řádku - ty taky ponecháme (pokud jsou validní); zase - tady nic o užitečnosti nerozhodujeme

In [4]:
# Shows analysis and all possible colors list combinations

print(parsed_data['colors'].describe(), end='\n\n')
with pd.option_context('display.max_rows', None):
    print(parsed_data['colors'].value_counts())

count       185
unique       43
top       Black
freq         49
Name: colors, dtype: object

colors
Black                      49
Blue                       25
Grey                       17
White                      16
Red                        14
Green                      13
Yellow                      4
Orange                      3
Aurora Blue                 3
Grey,Khaki,Green,Orange     2
Black,Red                   2
Haze Blue                   2
Gunmetal Black              2
Black Green                 2
Black,Yellow                2
Aurora Green                2
Yellow Graffiti             1
Black & Red                 1
Deep Blue                   1
Satin Black                 1
Pinkish Purple              1
Black,Orange                1
Mint Green                  1
Black & Yellow              1
Anthracite Gray             1
Meridian Black              1
Pearl White                 1
Black,White                 1
Black Red                   1
Silver White                1


### Implementujme výše popsané řešení

- počet řádku se musí zvětšit

In [5]:
# New column - `color` (one color per row)
parsed_data['color'] = pd.NA

print(f"Number of rows before transformation: {len(parsed_data)}", end='\n\n')
print(f"Number of NaN values before transformation: {parsed_data['colors'].isna().sum()}", end='\n\n')
# Make the rows without letters NaN - no valid color names
parsed_data.loc[(~parsed_data['colors'].str.contains(r'[a-zA-Z]',  na=False) | parsed_data['colors'].isna()), 'colors'] = pd.NA
print(f"Number of NaN values after partial transformation: {parsed_data['colors'].isna().sum()}", end='\n\n')

# Duplicate the rows for each color in the `colors` column
for _, row in parsed_data.dropna(subset=['colors']).iterrows():
    colors = row['colors'].split(',')
    first_was_assigned = False
    for color in colors:
        color = color.lower().strip()
        if not color:
            continue
        new_row = row.copy()
        new_row['color'] = color
        if not first_was_assigned:
            parsed_data.loc[row.name, 'color'] = color
            first_was_assigned = True
        else:
            parsed_data = pd.concat(
                [parsed_data, pd.DataFrame([new_row])],
                ignore_index=True
            )

# Get rid of the original column
parsed_data = parsed_data.drop('colors', axis=1)

print(f"Unique colors ({parsed_data['color'].nunique()} total): {parsed_data['color'].unique()}", end='\n\n')
print(f"Number of rows after transformation: {len(parsed_data)}", end='\n\n')
print(f"Number of NaN values after transformation: {parsed_data['color'].isna().sum()}", end='\n\n')
parsed_data.tail(3)


Number of rows before transformation: 320

Number of NaN values before transformation: 135

Number of NaN values after partial transformation: 136

Unique colors (37 total): [<NA> 'black' 'blue' 'yellow graffiti' 'grey' 'green' 'orange'
 'black & red' 'deep blue' 'white' 'red' 'satin black' 'aurora blue'
 'pinkish purple' 'haze blue' 'mint green' 'gunmetal black' 'black green'
 'black & yellow' 'anthracite gray' 'meridian black' 'pearl white'
 'aurora green' 'black red' 'silver white' 'black & gold' 'gray' 'golden'
 'yellow' 'dark green' 'silver' 'black & orange' 'black & blue' 'purple'
 'dark blue' 'matte black' 'blue grey' 'khaki']

Number of rows after transformation: 334

Number of NaN values after transformation: 138



,url,name,price,brand,motor_power_w,battery_capacity_wh,range_km,speed_kmh,color
331,https://www.geekbuying.com/item/79BIKE-Falcon-...,"79BIKE Falcon Pro Electric Motorcycle, Road le...",126943.50,79BIKE,5000,NaN,107.5,90.0,yellow
332,https://www.geekbuying.com/item/79BIKE-Falcon-...,"79BIKE Falcon Pro Electric Motorcycle, Road le...",126943.50,79BIKE,5000,", 2520Wh",95.0,90.0,purple
333,https://www.geekbuying.com/item/79BIKE-Falcon-...,"79BIKE Falcon Pro Electric Motorcycle, Road le...",126943.50,79BIKE,5000,",",2520.0,90.0,red


# 2. Battery capacity
- **Problém**: `battery_capacity_wh` zahrnuje nepřímá data
- **Řešení**: třeba vypočítat U*(I*t), konvertovat všechno na skutečná čísla. Pro takový různorodý rozsah vypracujeme víc pravidel na vynulování některých řádků, přidáme si i volnější imperická pravidla. Detailněji viz. níže

**Proč**: Potřebujeme čistá číselná data

**Proč neopravit chybu json formatu a nenaskénovat znovu?**: zajímavá praxe, data pocházejí ze samé reality - lidský faktor 

## Pravidla odstranění
1. Data můžou být ve formátu `(\d+(?:\.\d+)?)\s*[Vv].*?(\d+(?:\.\d+)?)\s*[Aa][Hh]` např. `52V 14.5Ah`
2. Data můžou být ve formátu `(\d+(?:\.\d+)?)\s*[Ww][Hh]` např. `2880 Wh`
3. Když najdeme několik grup v jednom řetězci - prioritu má první shoda (resp první shoda 2. formátu )
4. Definični obor: 0 < x < 1 se stává NaN

### Provedeme analýzu sloupce `battery_capacity_wh`

Vidíme hodně nekonzistencí. Proč? - chyba v LLM skénování:

návratový format sice má přísné json typování; když ale chybně zadáno - nezachrání  

náše chybná implementace:
```
"battery power capacity": {"type": "string"},
"range": {"type": "number"},
"speed": {"type": "number"}
```
správný format `battery power capacity` by měl být `number`

In [6]:
print(parsed_data['battery_capacity_wh'].describe(), end='\n\n')
with pd.option_context('display.max_rows', None):
    print(parsed_data['battery_capacity_wh'].value_counts())

count     325
unique    143
top       ,  
freq       82
Name: battery_capacity_wh, dtype: object

battery_capacity_wh
,                                                                                                                                                                                                                                                                                                                                                   82
,                                                                                                                                                                                                                                                                                                                                                   11
48V 18Ah                                                                                                                                                                                            

### Implementujme výše popsané řešení

je vidět anomalie např. kapacita mezi 0 a 1. Toto neháváme na podrobnější posouzení dalším zadáním

In [7]:
print(f"Number of rows before filtering: {len(parsed_data)}")
print(f"The row before modification: {parsed_data.iloc[0]['battery_capacity_wh']}", end='\n\n')

original = parsed_data['battery_capacity_wh']

# parse Wh
values_wh = original.astype(str).str.extract(
    r'(\d+(?:\.\d+)?)\s*[Ww][Hh]'
).astype(float)[0]

#values_wh = values_wh.where(~((values_wh > 0) & (values_wh < 1)), pd.NA) - no anomaly cleaning in this task

# parse Vh * Ah
values_ah = original.astype(str).str.extract(
    r'(\d+(?:\.\d+)?)\s*[Vv].*?(\d+(?:\.\d+)?)\s*[Aa][Hh]'
).astype(float)

values_wh_converted = round(values_ah[0] * values_ah[1], 2)

# combine parsed
parsed_data['battery_capacity_wh'] = values_wh.fillna(values_wh_converted)

# analyse
print(f"Number of rows after filtering: {len(parsed_data)}")
print(f"The row after modification: {parsed_data.iloc[0]['battery_capacity_wh']}", end='\n\n')
print(f"Number of null values: {parsed_data['battery_capacity_wh'].isnull().sum()}", end='\n\n')

print(parsed_data['battery_capacity_wh'].describe(), end='\n\n')
print(parsed_data['battery_capacity_wh'].value_counts(), end='\n\n')

parsed_data.head(3)

Number of rows before filtering: 334
The row before modification: 48 V, 20.8 Ah

Number of rows after filtering: 334
The row after modification: 998.4

Number of null values: 162

count     172.000000
mean      948.040698
std       641.038315
min         0.000000
25%       624.000000
50%       792.000000
75%      1040.000000
max      4176.000000
Name: battery_capacity_wh, dtype: float64

battery_capacity_wh
720.0     19
624.0     12
864.0     10
648.0      9
1040.0     7
960.0      6
540.0      6
499.2      6
840.0      5
1248.0     5
288.0      5
792.0      4
2880.0     4
468.0      4
2520.0     4
1196.0     3
1080.0     3
1008.0     3
1200.0     3
480.0      3
1440.0     3
518.4      3
998.4      2
2280.0     2
816.0      2
162.0      2
270.0      2
643.2      2
280.8      2
748.8      2
921.6      2
1800.0     2
1152.0     2
873.6      2
3120.0     2
1456.0     1
988.0      1
4176.0     1
576.0      1
1216.8     1
2160.0     1
54.6       1
828.0      1
672.0      1
754.0      1
1872

,url,name,price,brand,motor_power_w,battery_capacity_wh,range_km,speed_kmh,color
0,https://www.geekbuying.com/item/Ausom-L1-Elect...,"Ausom L1 Electric Scooter, 800W Motor, 48V 15....",10656.98,Ausom,500,998.4,90.0,20.0,<NA>
1,https://www.geekbuying.com/item/Kukirin-G2-Fol...,Kukirin G2 Foldable Electric Scooter 2026 Upgr...,11870.77,Kukirin,800,NaN,55.0,45.0,<NA>
2,https://www.geekbuying.com/item/BOYUEDA-Q7-Pro...,"BOYUEDA Q7 Pro Max Electric Scooter, 1600W*2 M...",18910.69,BOYUEDA,3200,1456.0,100.0,70.0,<NA>


# 3. Motor power

- **Problém**: sloupec `motor_power_w` - analýza
- **Řešení**: je to správně naskénovaná číselna data, věříme jim. Provedeme běžnou analýzu a ujistíme se, že jsou v pořádku

je vidět anomalie např. 750000 watt. Toto neháváme na podrobnější posouzení dalším zadáním

In [8]:
print(parsed_data['motor_power_w'].describe(), end='\n\n')
print(f"Number of null values: {parsed_data['motor_power_w'].isnull().sum()}", end='\n\n')

parsed_data['motor_power_w'] = parsed_data['motor_power_w'].astype(float)

print(parsed_data['motor_power_w'].describe(), end='\n\n')

count     331
unique     44
top       250
freq       74
Name: motor_power_w, dtype: object

Number of null values: 3

count       331.000000
mean       3622.187311
std       41341.950898
min           0.000000
25%         250.000000
50%         600.000000
75%        1000.000000
max      750000.000000
Name: motor_power_w, dtype: float64



# 4. Range

- **Problém**:  `range_km` sloupec - analýza
- **Řešení**: je to správně naskénovaná číselna data, věříme jim. Provedeme běžnou analýzu a ujistíme se, že jsou v pořádku


je vidět anomalie např. minimální délká je 0. Toto neháváme na podrobnější posouzení dalším zadáním

In [9]:
print(parsed_data['range_km'].describe(), end='\n\n')
print(f"Number of null values: {parsed_data['range_km'].isnull().sum()}", end='\n\n')

parsed_data['range_km'] = parsed_data['range_km'].astype(float)

print(parsed_data['range_km'].describe(), end='\n\n')

count     331.000000
mean       89.591541
std       196.870488
min         0.000000
25%        40.000000
50%        64.000000
75%       100.000000
max      2520.000000
Name: range_km, dtype: float64

Number of null values: 3

count     331.000000
mean       89.591541
std       196.870488
min         0.000000
25%        40.000000
50%        64.000000
75%       100.000000
max      2520.000000
Name: range_km, dtype: float64



# 5. Speed

- **Problém**: sloupec `speed_kmh` - analýza
- **Řešení**: je to správně naskénovaná číselna data, věříme jim. Provedeme běžnou analýzu a ujistíme se, že jsou v pořádku

je vidět anomalie např. 0 kmh. Toto neháváme na podrobnější posouzení dalším zadáním

In [10]:
print(parsed_data['speed_kmh'].describe(), end='\n\n')
print(f"Number of null values: {parsed_data['speed_kmh'].isnull().sum()}", end='\n\n')

parsed_data['speed_kmh'] = parsed_data['speed_kmh'].astype(float)

print(parsed_data['speed_kmh'].describe(), end='\n\n')

count    331.000000
mean      35.995468
std       24.108797
min        0.000000
25%       25.000000
50%       25.000000
75%       45.000000
max      250.000000
Name: speed_kmh, dtype: float64

Number of null values: 3

count    331.000000
mean      35.995468
std       24.108797
min        0.000000
25%       25.000000
50%       25.000000
75%       45.000000
max      250.000000
Name: speed_kmh, dtype: float64



# 6. Price

- **Problém**: sloupec `price` - analýza
- **Řešení**: je to správně naskénovaná číselna data, věříme jim. Provedeme běžnou analýzu a ujistíme se, že jsou v pořádku

anomalie nejsou vidět na první pohled

In [11]:
print(parsed_data['price'].describe(), end='\n\n')
print(f"Number of null values: {parsed_data['price'].isnull().sum()}", end='\n\n')

parsed_data['price'] = parsed_data['price'].astype(float)

print(parsed_data['price'].describe(), end='\n\n')

count          331
unique         148
top       26678.89
freq            14
Name: price, dtype: object

Number of null values: 3

count       331.000000
mean      31281.171450
std       26351.813734
min        3859.810000
25%       15633.480000
50%       26061.780000
75%       36389.130000
max      169904.920000
Name: price, dtype: float64



# 7. Price

- **Problém**: sloupec `name` - není potřeba další analýza. samotný sloupec je spíš informativní
- **Řešení**: není potřeba

nevíme co je tu anomalie

# 8. Kvalitativní data (brand)

- **Problém**: sloupec `brand`, provedeme běžnou analýzu, není to případ jako s `colors`.
- **Řešení**: není potřeba

anomalie nejsou vidět na první pohled

In [12]:
print(parsed_data['brand'].describe(), end='\n\n')
print(f"Number of null values: {parsed_data['brand'].isnull().sum()}", end='\n\n')

parsed_data['brand'] = (
    parsed_data['brand']
    .astype("string")
    .str.lower()
    .str.strip()
    .astype("category")
)

print(parsed_data['brand'].describe(), end='\n\n')

count         306
unique         55
top       Fafrees
freq           31
Name: brand, dtype: object

Number of null values: 28

count         306
unique         51
top       fafrees
freq           41
Name: brand, dtype: object



# 9. Finální kontrola

Zkontrolujeme si typy a konvertujeme explicitně

In [13]:
print(f"Before conversion: \n{parsed_data.dtypes}", end='\n\n')

parsed_data["url"] = parsed_data["url"].astype(str)
parsed_data["name"] = parsed_data["name"].astype(str)
parsed_data["price"] = parsed_data["price"].astype(float)
parsed_data["brand"] = parsed_data["brand"].astype("category")
parsed_data["motor_power_w"] = parsed_data["motor_power_w"].astype(float)
parsed_data["battery_capacity_wh"] = parsed_data["battery_capacity_wh"].astype(float)
parsed_data["range_km"] = parsed_data["range_km"].astype(float)
parsed_data["speed_kmh"] = parsed_data["speed_kmh"].astype(float)
parsed_data["color"] = parsed_data["color"].astype("category")

print(f"After conversion: \n{parsed_data.dtypes}", end='\n\n')

Before conversion: 
url                         str
name                        str
price                   float64
brand                  category
motor_power_w           float64
battery_capacity_wh     float64
range_km                float64
speed_kmh               float64
color                    object
dtype: object

After conversion: 
url                         str
name                        str
price                   float64
brand                  category
motor_power_w           float64
battery_capacity_wh     float64
range_km                float64
speed_kmh               float64
color                  category
dtype: object



# 10. Závěr

Podle zpětné zprávy z 1.projekyu, musela by tato část náležet prvnímu projektu. 
Rozhodli jsme ji proto dát do zvláštního 0-level modulu abychom se dále úplně věnovali užitečnému dolování

Původní data byla hodně zašumená, místy haotická a během úprav nám řekla moc i o meta stavu našich kódů. Ukázala na chyby a nepřesnosti. Snážili jsme se je zanalyzovat z pohledů nejprvé všeobecné objektivity a konzistence.


In [14]:
parsed_data.to_pickle("clean_data.pkl")